In [ ]:
#License: MIT

%matplotlib widget

import matplotlib.pylab as plt
import numpy as np
import os

import sidpy
import pyTEMlib
import pyTEMlib.file_tools
import pyTEMlib.image_tools


In [ ]:
folder='/path/to/folder/'

In [ ]:
#fileWidget.file_name
fname = '/path/fname.emd'



datasets = pyTEMlib.file_tools.open_file(fname)
#datasets = pyTEMlib.file_tools.open_file(fileWidget.file_name)#
selector = sidpy.ChooseDataset(datasets)



In [ ]:
selector.dataset

In [ ]:


dataset = selector.dataset
dataset.data_type = 'IMAGE_STACK'
dataset.x.dimension_type = 'SPATIAL'
dataset.y.dimension_type = 'SPATIAL'

view = dataset.plot()
dataset.x.slope



In [ ]:
dataset.shape

In [ ]:
#lets crop it to the fruitful area only

#dataset = dataset[:31]
dataset = dataset[:16]

In [ ]:
'''
Here we use the Diffeomorphic Demon Non-Rigid Registration as provided by simpleITK.
Please Cite:
    simpleITK
    and
    T. Vercauteren, X. Pennec, A. Perchant and N. Ayache Diffeomorphic Demons Using ITK's Finite Difference Solver Hierarchy The Insight Journal, 2007
'''

rigid_registered= pyTEMlib.image_tools.rigid_registration(dataset)
datasets['rigid_registered'] = rigid_registered
view = rigid_registered.plot()



In [ ]:
demon_registered = pyTEMlib.image_tools.demon_registration(rigid_registered)
datasets['demon_registered'] = demon_registered
view2 = demon_registered.plot()


In [ ]:
plt.close('all')
drift = rigid_registered.metadata['analysis']['rigid_registration']['drift']
polynom_degree = 2 # 1 is linear fit, 2 is parabolic fit, ...

x = np.linspace(0,drift.shape[0]-1,drift.shape[0])

line_fit_x = np.polyfit(x, drift[:,0], polynom_degree)
poly_x = np.poly1d(line_fit_x)
line_fit_y = np.polyfit(x, drift[:,1], polynom_degree)
poly_y = np.poly1d(line_fit_y)

plt.figure()
plt.axhline(color = 'gray')
plt.plot(x, drift[:,0], label = 'drift x')
plt.plot(x, drift[:,1], label = 'drift y')
plt.plot(x, poly_x(x),  label = 'fit_drift_x')
plt.plot(x, poly_y(x),  label = 'fit_drift_y')

plt.legend();
ax_pixels = plt.gca()
ax_pixels.step(1, 1)

scaleX = 1000 #(rigid_registered.x[1]-rigid_registered.x[0])*1000.  #in pm

ax_pm = ax_pixels.twinx()
x_1, x_2 = ax_pixels.get_ylim()

ax_pm.set_ylim(x_1*scaleX, x_2*scaleX)

ax_pixels.set_ylabel('drift [pixels]')
ax_pm.set_ylabel('drift [pm]')
ax_pixels.set_xlabel('image number');
plt.tight_layout()

In [ ]:
rigid_registered.metadata

In [ ]:
plt.close('all')
#plt.imshow(np.mean(demon_registered,axis=0)[75:325,50:1475])
plt.imshow(dataset)
plt.show()

In [ ]:
# save the averaged, registered frame as a calibrated dataset (carries pixel size),
# so atom_detect.ipynb can load it via pyTEMlib.file_tools.open_file
mean_image = demon_registered.mean(axis=0)
pyTEMlib.file_tools.save_dataset(mean_image, folder + 'aligned.hf5')
